# Semantic Embedding

Builds dense vector representations for all products using a two-tower architecture.

**Architecture:**
```
product_text_base  →  MPNet  →  product_emb  (768-dim)
                                                        →  combined_emb  →  FAISS index
review texts       →  MPNet  →  review_emb   (768-dim)
```

**Key design choices:**
| Choice | Reason |
|---|---|
| all-mpnet-base-v2 | STS benchmark 72.4 vs 68.1 for MiniLM — better semantic quality |
| Two-tower | Respects MPNet's 512-token limit; product and review signals stay separable |
| Weighted review average | 70% of raw reviews are 5-star — plain average is biased |
| Dynamic alpha per product | Products with weak review signal → product embedding dominates |
| Bias correction | Rare reviews (e.g. negative reviews in mostly-positive products) get boosted |
| IndexFlatIP | Exact search; < 5ms for ~7k products; no approximation needed |

**Output files (all in DATA_DIR):**
- `product_embeddings.npy` — (n_products, 768)
- `review_embeddings.npy`  — (n_products, 768)
- `combined_embeddings.npy` — (n_products, 768)
- `faiss_index.bin` — FAISS IndexFlatIP
- `embedding_index.csv` — maps row index → product metadata


## Setup

In [1]:
import sys
sys.path.insert(0, "src")

import os, time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
import faiss

from mean_squared_terrors.config import (
    MODEL_NAME, BATCH_SIZE, W_HELPFUL, W_RATING, ALPHA_MIN, ALPHA_MAX,
)
from mean_squared_terrors.embedding import (
    compute_review_weights, compute_review_embedding,
    compute_dynamic_alpha, blend_embeddings, faiss_search,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

if torch.backends.mps.is_available():    DEVICE = "mps"
elif torch.cuda.is_available():           DEVICE = "cuda"
else:                                     DEVICE = "cpu"
print(f"Device: {DEVICE}")

ModuleNotFoundError: No module named 'faiss'

## Configuration

Set `DATA_DIR` to your data folder. All other parameters are documented inline.
On Google Colab with T4 GPU, uncomment the Drive mount lines and update DATA_DIR.


In [ ]:
DATA_DIR      = "data"  # update this path if needed
PRODUCTS_PATH = os.path.join(DATA_DIR, "products_cleaned.csv")
REVIEWS_PATH  = os.path.join(DATA_DIR, "reviews_cleaned.csv")

# All other config (MODEL_NAME, BATCH_SIZE, W_HELPFUL, W_RATING, ALPHA_MIN, ALPHA_MAX,
# MIN_REVIEW_TOKENS) is imported from src/mean_squared_terrors/config.py
print("Config OK")
print(f"  Model   : {MODEL_NAME}")
print(f"  Alpha   : dynamic [{ALPHA_MIN}, {ALPHA_MAX}]")
print(f"  Weights : helpful={W_HELPFUL}, rating_dist={W_RATING} + bias correction")
print(f"  Min review tokens: {MIN_REVIEW_TOKENS}")

## Load Data

In [ ]:
products = pd.read_csv(PRODUCTS_PATH)
reviews  = pd.read_csv(REVIEWS_PATH)

reviews = reviews[reviews["text"].fillna("").str.split().str.len() >= MIN_REVIEW_TOKENS].reset_index(drop=True)

common   = set(products["parent_asin"]) & set(reviews["parent_asin"])
products = products[products["parent_asin"].isin(common)].reset_index(drop=True)
reviews  = reviews[reviews["parent_asin"].isin(common)].reset_index(drop=True)

print(f"Products : {len(products):,}")
print(f"Reviews  : {len(reviews):,}")

asin_to_idx = {asin: i for i, asin in enumerate(products["parent_asin"])}

## Model Initialization

In [ ]:
print(f"Loading model: {MODEL_NAME}")
t0    = time.time()
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
DIM   = model.get_sentence_embedding_dimension()
print(f"Ready in {time.time()-t0:.1f}s  |  embedding dim={DIM}")


Loading model: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7632.07it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready in 41.3s  |  embedding dim=768


## Product Embeddings

Encodes `product_text_base` for all 11,537 products.
`normalize_embeddings=True` produces L2-normalised vectors so that
inner product equals cosine similarity — required for FAISS IndexFlatIP.


In [ ]:
print("Encoding products...")
t0 = time.time()
product_emb = model.encode(
    products["product_text_base"].fillna("").astype(str).tolist(),
    batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"\nDone in {time.time()-t0:.1f}s  |  shape: {product_emb.shape}")
print(f"Mean norm: {np.linalg.norm(product_emb, axis=1).mean():.4f}")


Encoding products...


Batches: 100%|██████████| 119/119 [16:17<00:00,  8.21s/it]



Done in 977.5s  |  shape: (7604, 768)
Mean norm: 1.0000


## Review Embeddings — Weighted Average

**Step A:** encode each review individually  
**Step B:** compute per-review weights  
**Step C:** compute weighted average per product

**Weight formula:**
```
weight_i = 0.7 × log1p(helpful_vote_i) / max(log1p) 
         + 0.3 × |rating_i - 3| / 2
         × bias_correction_i
```

- `helpful_vote` component: community-validated quality signal. log1p prevents
  a review with 5000 votes from dominating all others.
- `rating distance` component: reviews at rating extremes (1 or 5) carry more
  information than neutral reviews (3 stars).
- `bias_correction`: reviews that are rare for their product (e.g. a negative
  review in a mostly-positive product) get a boost factor of up to 3.0×.


In [ ]:
print("Encoding individual reviews...")
t0 = time.time()
review_emb_raw = model.encode(
    reviews["text"].fillna("").astype(str).tolist(),
    batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"Done in {time.time()-t0:.1f}s  |  shape: {review_emb_raw.shape}")


Encoding individual reviews...


Batches:   1%|          | 22/2048 [10:59<16:52:31, 29.99s/it]


KeyboardInterrupt: 

In [ ]:
weights_final = compute_review_weights(reviews)

print(f"Final weights — min: {weights_final.min():.4f}  max: {weights_final.max():.4f}  mean: {weights_final.mean():.4f}")

# Weight distribution plot
hv      = reviews["helpful_vote"].fillna(0).values.astype(float)
hv_norm = np.log1p(hv) / (np.log1p(hv).max() or 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
fig.suptitle("Review weight distribution (with bias correction)", fontweight="bold")
axes[0].hist(hv_norm, bins=50, color="steelblue"); axes[0].set_title("helpful_vote norm")
axes[1].hist(weights_final, bins=50, color="seagreen"); axes[1].set_title("final weight")
plt.tight_layout(); plt.show()

In [ ]:
review_emb = compute_review_embedding(review_emb_raw, reviews, weights_final, asin_to_idx, DIM)

print(f"Review embedding — shape: {review_emb.shape}")
print(f"Mean norm: {np.linalg.norm(review_emb, axis=1).mean():.4f}")

## Dynamic Alpha per Product

Alpha controls the blend between product embedding and review embedding:
```
combined = alpha × product_emb + (1 - alpha) × review_emb
```

Alpha is computed per product based on review signal quality:
- **n_reviews** (normalised, 3→16): more reviews = stronger signal
- **max_helpful_vote** (normalised, 0→20): community validation

Products with `alpha > 0.65` have weak review signals → product text dominates.  
Products with `alpha < 0.45` have strong review signals → review embedding matters more.


In [ ]:
asin_alpha = compute_dynamic_alpha(reviews, products)
alphas = np.array(list(asin_alpha.values()))

print(f"Dynamic alpha per product:")
print(f"  mean={alphas.mean():.3f}  min={alphas.min():.3f}  max={alphas.max():.3f}")
print(f"  Products with alpha < 0.45 (review dominates): {(alphas < 0.45).mean():.1%}")
print(f"  Products with alpha > 0.60 (product dominates):  {(alphas > 0.60).mean():.1%}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(alphas, bins=40, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(alphas.mean(), color="red", ls="--", lw=1.5, label=f"mean={alphas.mean():.3f}")
ax.set(title="Dynamic alpha distribution per product",
       xlabel="Alpha (high = product emb dominates)", ylabel="# Products")
ax.legend()
plt.tight_layout(); plt.show()

## Combined Embedding

In [ ]:
combined_emb = blend_embeddings(product_emb, review_emb, asin_alpha, products)

print(f"Combined embedding — shape: {combined_emb.shape}")
print(f"Mean norm: {np.linalg.norm(combined_emb, axis=1).mean():.4f}")

## Quality Check

Two checks:
- **PCA 2D**: compress 768 dims to 2 for visualisation — coloured by avg rating
- **Cosine similarity between random pairs**: should be centred near 0,
  meaning products are well-separated in the vector space


In [ ]:
from sklearn.decomposition import PCA

SAMPLE = min(2000, len(combined_emb))
rng    = np.random.RandomState(42)
sidx   = rng.choice(len(combined_emb), SAMPLE, replace=False)
pca    = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(combined_emb[sidx])
ratings = products["product_avg_rating"].values[sidx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Embedding quality check", fontweight="bold")

sc = axes[0].scatter(emb_2d[:,0], emb_2d[:,1], c=ratings,
                     cmap="RdYlGn", alpha=0.4, s=10, vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[0], label="Avg rating")
axes[0].set_title(f"PCA 2D projection (n={SAMPLE})")
axes[0].set_xlabel(f"PC1 — {pca.explained_variance_ratio_[0]*100:.1f}% var")
axes[0].set_ylabel(f"PC2 — {pca.explained_variance_ratio_[1]*100:.1f}% var")

n_pairs = 5000
ia = rng.randint(0, len(combined_emb), n_pairs)
ib = rng.randint(0, len(combined_emb), n_pairs)
cos = (combined_emb[ia] * combined_emb[ib]).sum(axis=1)
axes[1].hist(cos, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[1].axvline(cos.mean(), color="red", ls="--", lw=1.5, label=f"mean={cos.mean():.3f}")
axes[1].set_title("Cosine similarity — random pairs")
axes[1].set_xlabel("Cosine similarity")
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Variance explained PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")
print(f"Mean cosine similarity:   {cos.mean():.3f}")


## FAISS Index

Builds an exact inner product index over the combined embeddings.
Self-retrieval test: the first product should find itself at rank 1 with score 1.0.


In [ ]:
index = faiss.IndexFlatIP(DIM)
index.add(combined_emb)
print(f"FAISS index — {index.ntotal:,} vectors  |  dim={DIM}")

D, I = index.search(combined_emb[0:1], k=6)
print(f"\nSelf-retrieval test — '{products['product_title'].iloc[0][:55]}...'")
for rank, (score, idx) in enumerate(zip(D[0], I[0])):
    marker = "  <- itself" if idx == 0 else ""
    print(f"  {rank+1}. [{score:.4f}]  {str(products['product_title'].iloc[idx])[:55]}{marker}")


## Saving Outputs

Saves embeddings as numpy arrays and the FAISS index as a binary file.
Also saves `embedding_index.csv` which maps each FAISS row to product metadata.


In [ ]:
np.save(os.path.join(DATA_DIR, "product_embeddings.npy"),  product_emb)
np.save(os.path.join(DATA_DIR, "review_embeddings.npy"),   review_emb)
np.save(os.path.join(DATA_DIR, "combined_embeddings.npy"), combined_emb)
faiss.write_index(index, os.path.join(DATA_DIR, "faiss_index.bin"))

# Save per-product alpha — useful for analysis and debugging
alpha_df = pd.DataFrame({
    "parent_asin": list(asin_alpha.keys()),
    "alpha":       list(asin_alpha.values())
})

bins   = [0, 10, 25, 50, 100, float("inf")]
labels = ["budget","low","mid","high","premium"]
index_df = products[["parent_asin","product_title","brand",
                       "product_avg_rating","product_rating_count","price"]].copy()
index_df = index_df.merge(alpha_df, on="parent_asin", how="left")
index_df["emb_row"]      = range(len(index_df))
index_df["price_bucket"] = pd.cut(index_df["price"], bins=bins, labels=labels)
index_df.to_csv(os.path.join(DATA_DIR, "embedding_index.csv"), index=False)

print("Files saved:")
for fname in ["product_embeddings.npy","review_embeddings.npy",
               "combined_embeddings.npy","faiss_index.bin","embedding_index.csv"]:
    p = os.path.join(DATA_DIR, fname)
    print(f"  {fname:<38}  {os.path.getsize(p)/1e6:.1f} MB")


## Search Test

Quick end-to-end test of the retrieval pipeline.
Use this to verify that the embeddings are working before running notebook 04.


In [ ]:
index_df_loaded = pd.read_csv(os.path.join(DATA_DIR, "embedding_index.csv"))

queries = [
    ("affordable moisturizer for sensitive skin", None),
    ("air purifier quiet bedroom allergies",       None),
    ("cheap sunscreen SPF 50",                    "budget"),
    ("vitamin C supplement immune system",         None),
    ("electric toothbrush whitening",              None),
]

for query, bucket in queries:
    price_filter = f"  [price: {bucket}]" if bucket else ""
    print(f"\n{'─'*68}")
    print(f"Query: '{query}'{price_filter}")
    print('─'*68)
    res = faiss_search(query, model, index, index_df_loaded, k=5, price_bucket=bucket)
    for i, (_, row) in enumerate(res.iterrows()):
        price_str = f"${row['price']:.2f}" if pd.notna(row['price']) else "  N/A"
        alpha  = f"a={row['alpha']:.2f}" if pd.notna(row.get('alpha')) else ""
        title = str(row["product_title"])[:50]
        print(f"  {i+1}. [{row['score']:.4f}] ⭐{row['product_avg_rating']:.1f}"
              f"  {price_str:>8}  {alpha}  {title}")